# State Trajectory Analysis — Layer V ET Identity

## Gene States (per timepoint per cell type)
```
State 1: RNA high + Chromatin Open   → Actively expressed
State 2: RNA high + Chromatin Closed → FORCED EXPRESSION (TF/ncRNA driving)
State 3: RNA low  + Chromatin Open   → FORCED SUPPRESSION (TF/ncRNA blocking)
State 4: RNA low  + Chromatin Closed → Silenced
```

## Trajectory Comparison
- Cortex  (3 timepoints): Hamming distance ≤ 1 (1/3 tolerance)
- Spinal cord (7 timepoints): Hamming distance ≤ 2 (2/7 tolerance)

## Narrowing Logic
- Same trajectory in ALL cell types → ubiquitous, not identity-specific
- Same only in neurons → neuron-identity
- Same only in deep layer neurons → deep layer identity
- Unique to Layer V ET → Layer V ET-specific identity-forming

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import product
from collections import defaultdict

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 60)

## 0. Configuration

In [ ]:
# ── OUTPUT from alignment pipelines ──────────────────────────────────────────
CORTEX_RNA_DIR    = Path("/home/nakagawa/datasets/aligned_cortex")
SPINAL_RNA_DIR    = Path("/home/nakagawa/datasets/aligned_spinalcord/rna")
SPINAL_ATAC_DIR   = Path("/home/nakagawa/datasets/aligned_spinalcord/atac")

# GTF for gene-to-peak annotation
GTF_PATH = "/home/nakagawa/datasets/genome/mm10/Mus_musculus.GRCm38.84.gtf"

# Output
OUT_DIR = Path("/home/nakagawa/datasets/state_trajectory_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Thresholds ────────────────────────────────────────────────────────────────
PERCENTILES    = [50, 75]        # sensitivity check: both run, results saved separately
HAMMING_CORTEX = 1               # ≤1 mismatch out of 3 timepoints
HAMMING_SPINAL = 2               # ≤2 mismatches out of 7 timepoints

# ── Cell type groups — edit after seeing what's in your count matrices ────────
LAYER_V_ET = "L5_ET"             # exact label used in your data

NON_NEURON_TYPES = [
    "Astrocyte", "Microglia", "Oligodendrocyte", "OPC", "Endothelial", "VLMC"
]
UPPER_LAYER_TYPES = [
    "L2_3_IT", "L4_IT"
]
OTHER_DEEP_LAYER_TYPES = [
    "L5_IT", "L5_6_NP", "L6_IT", "L6_CT", "L6b"
]

# ── Timepoints — must match your sample labels from alignment scripts ─────────
# Cortex: 3 overlapping RNA+ATAC timepoints
CORTEX_TIMEPOINTS = ["E13.5", "E15.5", "E18.5"]

# Spinal cord: 7 paired RNA+ATAC timepoints
SPINAL_TIMEPOINTS = ["E10.5", "E13.5", "P4", "P13", "P21", "P56", "2yr"]

print("Configuration set.")
print(f"Percentile thresholds : {PERCENTILES}")
print(f"Hamming cortex        : ≤{HAMMING_CORTEX} / {len(CORTEX_TIMEPOINTS)}")
print(f"Hamming spinal        : ≤{HAMMING_SPINAL} / {len(SPINAL_TIMEPOINTS)}")

## 1. Helper Functions

In [ ]:
def hamming_distance(s1, s2):
    """Number of positions where two equal-length state trajectories differ."""
    assert len(s1) == len(s2), "Trajectories must be same length"
    return sum(a != b for a, b in zip(s1, s2))


def assign_state(rna_high, atac_open):
    """
    Assign state 1-4 from boolean RNA and ATAC calls.
    State 1: high RNA + open  chromatin
    State 2: high RNA + closed chromatin  ← FORCED EXPRESSION
    State 3: low  RNA + open  chromatin   ← FORCED SUPPRESSION
    State 4: low  RNA + closed chromatin
    """
    if     rna_high and     atac_open: return 1
    if     rna_high and not atac_open: return 2
    if not rna_high and     atac_open: return 3
    if not rna_high and not atac_open: return 4


def trajectories_match(traj_a, traj_b, max_hamming):
    """True if two trajectory tuples are within Hamming distance."""
    return hamming_distance(traj_a, traj_b) <= max_hamming


def classify_cell_type(ct, layer_v_et, non_neuron, upper_layer, deep_layer):
    if ct == layer_v_et:      return "LayerVET"
    if ct in non_neuron:      return "NonNeuron"
    if ct in upper_layer:     return "UpperLayer"
    if ct in deep_layer:      return "DeepLayer"
    return "Other"


print("Helper functions defined.")

## 2. Load RNA Count Matrices

STAR `--quantMode GeneCounts` outputs `ReadsPerGene.out.tab`  
STARsolo outputs `Solo.out/GeneFull/filtered/matrix.mtx.gz`  
We handle both formats below.

In [ ]:
import scipy.io
import gzip

def load_starsolo_counts(sample_dir):
    """
    Load STARsolo filtered count matrix.
    Returns DataFrame: genes x cells.
    """
    mtx_path = Path(sample_dir) / "Solo.out/GeneFull/filtered"
    mat      = scipy.io.mmread(mtx_path / "matrix.mtx.gz").T.toarray()

    with gzip.open(mtx_path / "features.tsv.gz", "rt") as f:
        genes = [line.strip().split("\t")[1] for line in f]  # gene name col
    with gzip.open(mtx_path / "barcodes.tsv.gz", "rt") as f:
        barcodes = [line.strip() for line in f]

    return pd.DataFrame(mat, index=barcodes, columns=genes)  # cells x genes


def load_star_bulk_counts(tab_path):
    """
    Load STAR ReadsPerGene.out.tab (bulk / snRNA paired-end).
    Column 2 = unstranded, col 3 = forward, col 4 = reverse.
    SMARTer Stranded → use column 4 (reverse strand).
    Returns Series: gene → count.
    """
    df = pd.read_csv(tab_path, sep="\t", header=None,
                     names=["gene", "unstranded", "forward", "reverse"],
                     skiprows=4)   # skip N_unmapped etc summary rows
    return df.set_index("gene")["reverse"]


# ── Load cortex 10X scRNA-seq (STARsolo) ─────────────────────────────────────
# We pseudobulk per cell type per timepoint
# Cell type labels come from your h5ad annotation (transfer via barcode matching)
# For now we load raw matrices; cell type assignment is in Cell 3 below

print("Scanning cortex STARsolo outputs...")
cortex_samples = {}
for sample_dir in sorted(CORTEX_RNA_DIR.iterdir()):
    if not sample_dir.is_dir(): continue
    mtx = sample_dir / "Solo.out/GeneFull/filtered/matrix.mtx.gz"
    if mtx.exists():
        tp = None
        for t in CORTEX_TIMEPOINTS:
            if t.replace(".", "") in sample_dir.name or t in sample_dir.name:
                tp = t
                break
        if tp:
            cortex_samples[tp] = sample_dir
            print(f"  Found: {sample_dir.name} → {tp}")

print(f"\nCortex timepoints available: {list(cortex_samples.keys())}")
missing = [t for t in CORTEX_TIMEPOINTS if t not in cortex_samples]
if missing:
    print(f"[WARNING] Missing timepoints: {missing} — alignment may still be running")

# ── Load spinal cord bulk snRNA-seq (STAR ReadsPerGene) ──────────────────────
print("\nScanning spinal cord RNA outputs...")
spinal_rna_samples = {}
for sample_dir in sorted(SPINAL_RNA_DIR.iterdir()):
    if not sample_dir.is_dir(): continue
    tab = sample_dir / "ReadsPerGene.out.tab"
    if tab.exists():
        tp = None
        for t in SPINAL_TIMEPOINTS:
            t_safe = t.replace(".", "")
            if t_safe in sample_dir.name or t in sample_dir.name:
                tp = t
                break
        if tp:
            if tp not in spinal_rna_samples:
                spinal_rna_samples[tp] = []
            spinal_rna_samples[tp].append(tab)
            print(f"  Found: {sample_dir.name} → {tp}")

print(f"\nSpinal cord timepoints available: {list(spinal_rna_samples.keys())}")

## 3. Transfer Cell Type Labels to Cortex scRNA-seq
Match barcodes from STARsolo output to h5ad annotations.

In [ ]:
import scanpy as sc

# ── EDIT: path to your annotated h5ad ────────────────────────────────────────
H5AD_PATH  = "/home/nakagawa/datasets/h5ad/10X_cells_v3_AIBS.h5ad"
CT_COL     = "cell_type"   # adjust if needed after checking adata.obs.columns
# ─────────────────────────────────────────────────────────────────────────────

print("Loading h5ad for barcode→cell type mapping...")
adata_ref = sc.read_h5ad(H5AD_PATH)
print(f"  Cells: {adata_ref.n_obs}")
print(f"  Cell types: {adata_ref.obs[CT_COL].nunique()}")

# Build barcode → cell type lookup
# Barcodes in h5ad may have sample suffix (e.g. ACGT-1); strip suffix
barcode_to_ct = (
    adata_ref.obs[CT_COL]
    .reset_index()
    .rename(columns={"index": "barcode"})
)
barcode_to_ct["barcode_clean"] = barcode_to_ct["barcode"].str.split("-").str[0]
bc_map = dict(zip(barcode_to_ct["barcode_clean"], barcode_to_ct[CT_COL]))
print(f"  Barcode map built: {len(bc_map)} entries")

# ── Build pseudobulk RNA per cell type per timepoint (cortex) ────────────────
# pseudobulk = sum of counts across cells of same type, then CPM normalize

print("\nBuilding cortex pseudobulk matrices...")
cortex_pseudobulk = {}   # {timepoint: DataFrame(genes x cell_types)}

for tp, sample_dir in cortex_samples.items():
    print(f"  Loading {tp}...")
    counts = load_starsolo_counts(sample_dir)   # cells x genes

    # Map barcodes to cell types
    counts.index = counts.index.str.split("-").str[0]   # strip suffix
    counts["cell_type"] = counts.index.map(bc_map)
    counts = counts.dropna(subset=["cell_type"])

    mapped_pct = len(counts) / load_starsolo_counts(sample_dir).shape[0] * 100
    print(f"    Mapped {len(counts)} / {load_starsolo_counts(sample_dir).shape[0]} cells ({mapped_pct:.1f}%)")

    # Pseudobulk: sum per cell type
    gene_cols = [c for c in counts.columns if c != "cell_type"]
    pb = counts.groupby("cell_type")[gene_cols].sum().T   # genes x cell_types

    # CPM normalize
    pb = pb.div(pb.sum(axis=0), axis=1) * 1e6
    cortex_pseudobulk[tp] = pb
    print(f"    Pseudobulk shape: {pb.shape}  (genes x cell types)")

print("\nCortex pseudobulk done.")

## 4. Build Spinal Cord Pseudobulk RNA
Bulk snRNA — average replicates per timepoint, CPM normalize.

In [ ]:
# Spinal cord is bulk snRNA — no single-cell barcode demux needed
# Each SRR = one replicate of one timepoint
# We average replicates → one expression vector per timepoint
# NOTE: spinal cord has no single-cell resolution → trajectory is tissue-level
# Cell type specificity here comes from the ChatCre/HB9GFP cell-type-specific
# nuclear isolation (INTACT protocol) — each SRR is already cell-type enriched

print("Building spinal cord pseudobulk RNA...")
spinal_rna_pseudobulk = {}   # {timepoint: Series(genes)}

for tp, tab_list in spinal_rna_samples.items():
    rep_counts = []
    for tab in tab_list:
        s = load_star_bulk_counts(tab)
        rep_counts.append(s)

    # Average across replicates
    merged = pd.concat(rep_counts, axis=1).fillna(0)
    mean_counts = merged.mean(axis=1)

    # CPM normalize
    cpm = mean_counts / mean_counts.sum() * 1e6
    spinal_rna_pseudobulk[tp] = cpm
    print(f"  {tp}: {len(tab_list)} replicates, {len(cpm)} genes")

# Combine into genes x timepoints matrix
spinal_rna_matrix = pd.DataFrame(spinal_rna_pseudobulk)   # genes x timepoints
spinal_rna_matrix = spinal_rna_matrix.fillna(0)
print(f"\nSpinal cord RNA matrix: {spinal_rna_matrix.shape}  (genes x timepoints)")

## 5. Load ATAC-seq Peak Data
Summarize ATAC signal per gene using peaks overlapping gene bodies ± 2kb promoter.

In [ ]:
def load_narrowpeak(path):
    """Load MACS2 narrowPeak file. Returns DataFrame with chr, start, end, score."""
    cols = ["chr","start","end","name","score","strand",
            "signalValue","pValue","qValue","peak"]
    return pd.read_csv(path, sep="\t", header=None, names=cols)


def parse_gtf_genes(gtf_path, extend_promoter=2000):
    """
    Parse GTF to get gene body + promoter windows.
    Returns DataFrame: gene_name, chr, start, end (with promoter extension).
    """
    print("  Parsing GTF for gene windows...")
    rows = []
    with open(gtf_path) as f:
        for line in f:
            if line.startswith("#"): continue
            parts = line.strip().split("\t")
            if parts[2] != "gene": continue
            info = parts[8]
            name = None
            for field in info.split(";"):
                field = field.strip()
                if field.startswith("gene_name"):
                    name = field.split('"')[1]
                    break
            if name is None: continue
            chrom  = parts[0]
            start  = max(0, int(parts[3]) - 1 - extend_promoter)
            end    = int(parts[4]) + extend_promoter
            strand = parts[6]
            rows.append({"gene": name, "chr": chrom,
                         "start": start, "end": end, "strand": strand})
    return pd.DataFrame(rows).drop_duplicates(subset="gene")


def peaks_to_gene_scores(peaks_df, gene_windows):
    """
    For each gene, sum signalValue of peaks overlapping gene window.
    Returns Series: gene → ATAC score.
    """
    gene_scores = {}
    for _, gw in gene_windows.iterrows():
        overlapping = peaks_df[
            (peaks_df["chr"] == gw["chr"]) &
            (peaks_df["start"] < gw["end"]) &
            (peaks_df["end"]   > gw["start"])
        ]
        gene_scores[gw["gene"]] = overlapping["signalValue"].sum()
    return pd.Series(gene_scores)


# Parse GTF once
print("Parsing GTF gene windows...")
gene_windows = parse_gtf_genes(GTF_PATH, extend_promoter=2000)
print(f"  Gene windows: {len(gene_windows)}")

# Load and process spinal cord ATAC
print("\nLoading spinal cord ATAC peaks...")
spinal_atac_scores = {}   # {timepoint: Series(gene → score)}

for sample_dir in sorted(SPINAL_ATAC_DIR.iterdir()):
    if not sample_dir.is_dir(): continue
    peaks_files = list(sample_dir.glob("*_peaks.narrowPeak"))
    if not peaks_files: continue

    # Identify timepoint from directory name
    tp = None
    for t in SPINAL_TIMEPOINTS:
        t_safe = t.replace(".", "")
        if t_safe in sample_dir.name or t in sample_dir.name:
            tp = t
            break
    if tp is None: continue

    peaks_df = load_narrowpeak(peaks_files[0])
    scores   = peaks_to_gene_scores(peaks_df, gene_windows)

    if tp not in spinal_atac_scores:
        spinal_atac_scores[tp] = []
    spinal_atac_scores[tp].append(scores)
    print(f"  {sample_dir.name} → {tp}: {len(peaks_df)} peaks")

# Average replicates
spinal_atac_matrix = pd.DataFrame({
    tp: pd.concat(reps, axis=1).fillna(0).mean(axis=1)
    for tp, reps in spinal_atac_scores.items()
}).fillna(0)

print(f"\nSpinal cord ATAC matrix: {spinal_atac_matrix.shape}  (genes x timepoints)")

## 6. Assign States at Each Timepoint
Run for both 50th and 75th percentile thresholds.

In [ ]:
def assign_states_matrix(rna_mat, atac_mat, timepoints, percentile):
    """
    For each gene at each timepoint, assign state 1-4.
    rna_mat  : genes x timepoints (or genes x cell_types for cortex)
    atac_mat : genes x timepoints
    Returns  : genes x timepoints DataFrame of states (int 1-4)
    """
    # Align gene index
    common_genes = rna_mat.index.intersection(atac_mat.index)
    rna  = rna_mat.loc[common_genes]
    atac = atac_mat.loc[common_genes]

    state_mat = pd.DataFrame(index=common_genes, columns=timepoints, dtype=int)

    for tp in timepoints:
        if tp not in rna.columns or tp not in atac.columns:
            state_mat[tp] = np.nan
            continue

        rna_thresh  = np.percentile(rna[tp].dropna(),  percentile)
        atac_thresh = np.percentile(atac[tp].dropna(), percentile)

        rna_high  = rna[tp]  >= rna_thresh
        atac_open = atac[tp] >= atac_thresh

        state_mat[tp] = [
            assign_state(rh, ao)
            for rh, ao in zip(rna_high, atac_open)
        ]

    return state_mat


# ── Cortex: assign states per cell type per timepoint ────────────────────────
# For cortex we have pseudobulk per cell type at each timepoint
# We need to build a combined RNA matrix: genes x (celltype_timepoint)
# then assign states per cell type separately

print("Assigning states...")
cortex_states   = {}   # {percentile: {cell_type: state_trajectory_Series}}
spinal_states   = {}   # {percentile: {timepoint: state_Series}}

# Get all cell types present across all cortex timepoints
all_cortex_cts = set()
for pb in cortex_pseudobulk.values():
    all_cortex_cts.update(pb.columns)
print(f"Cortex cell types found: {sorted(all_cortex_cts)}")

for pct in PERCENTILES:
    print(f"\n--- Percentile threshold: {pct}th ---")
    cortex_states[pct]  = {}
    spinal_states[pct]  = {}

    # Cortex: per cell type
    for ct in all_cortex_cts:
        # Build RNA matrix for this cell type across timepoints
        rna_ct_tp = pd.DataFrame({
            tp: pb[ct]
            for tp, pb in cortex_pseudobulk.items()
            if ct in pb.columns
        })
        if rna_ct_tp.empty: continue

        # Use cortex ATAC if available — add cortex ATAC loading here if present
        # For now use spinal_atac_matrix as proxy for timepoints E13.5, E15.5, E18.5
        # NOTE: replace with cortex ATAC when aligned (SRR12082772-774)
        atac_proxy = spinal_atac_matrix.reindex(rna_ct_tp.index).fillna(0)
        available_tps = [t for t in CORTEX_TIMEPOINTS if t in rna_ct_tp.columns]

        states = assign_states_matrix(rna_ct_tp, atac_proxy, available_tps, pct)
        cortex_states[pct][ct] = states

    # Spinal cord: tissue-level (one RNA+ATAC track per timepoint)
    spinal_states[pct] = assign_states_matrix(
        spinal_rna_matrix, spinal_atac_matrix, SPINAL_TIMEPOINTS, pct
    )
    n_state2 = (spinal_states[pct] == 2).sum().sum()
    n_state3 = (spinal_states[pct] == 3).sum().sum()
    print(f"  Spinal: State2 (forced expr)={n_state2}  State3 (forced supp)={n_state3}")

print("\nState assignment done.")

## 7. Extract State Trajectories
Each gene gets a trajectory tuple, e.g. (4, 4, 2, 1, 1, 1, 1) across timepoints.

In [ ]:
def get_trajectories(state_mat, timepoints):
    """
    Convert state matrix to dict: gene → trajectory tuple.
    Drops genes with any NaN timepoint.
    """
    valid_tp = [t for t in timepoints if t in state_mat.columns]
    sub = state_mat[valid_tp].dropna()
    return {
        gene: tuple(int(v) for v in row)
        for gene, row in sub.iterrows()
    }


# Build trajectory dicts
# cortex_traj[pct][cell_type] = {gene: (s1, s2, s3)}
# spinal_traj[pct]            = {gene: (s1,...,s7)}

cortex_traj = {}
spinal_traj = {}

for pct in PERCENTILES:
    cortex_traj[pct] = {}
    for ct, state_mat in cortex_states[pct].items():
        cortex_traj[pct][ct] = get_trajectories(state_mat, CORTEX_TIMEPOINTS)

    spinal_traj[pct] = get_trajectories(spinal_states[pct], SPINAL_TIMEPOINTS)

# Quick preview
pct = PERCENTILES[0]
if LAYER_V_ET in cortex_traj[pct]:
    traj_dict = cortex_traj[pct][LAYER_V_ET]
    print(f"Example cortex Layer V ET trajectories (pct={pct}):")
    for gene, traj in list(traj_dict.items())[:10]:
        print(f"  {gene:<20} {traj}")
else:
    print(f"[NOTE] {LAYER_V_ET} not found in cortex cell types.")
    print(f"Available: {list(cortex_traj[pct].keys())}")
    print("Update LAYER_V_ET in Cell 0 to match exactly.")

## 8. Find Genes with Layer V ET-Like Trajectories in Other Cell Types
Using proportional Hamming distance tolerance.

In [ ]:
def find_matching_cell_types(gene, lvet_traj, all_trajs, max_hamming):
    """
    For a given gene and its Layer V ET trajectory,
    return list of cell types whose trajectory matches within max_hamming.
    all_trajs: {cell_type: {gene: traj}}
    """
    matches = []
    for ct, traj_dict in all_trajs.items():
        if gene not in traj_dict: continue
        if hamming_distance(lvet_traj, traj_dict[gene]) <= max_hamming:
            matches.append(ct)
    return matches


def categorize_specificity(matching_cts, layer_v_et,
                            non_neuron, upper_layer, deep_layer):
    """
    Given which cell types share a trajectory with Layer V ET,
    return specificity tier.
    """
    others = [ct for ct in matching_cts if ct != layer_v_et]
    if not others:
        return "Tier3_LayerVET_Specific"

    has_non_neuron  = any(ct in non_neuron  for ct in others)
    has_upper_layer = any(ct in upper_layer for ct in others)
    has_deep_layer  = any(ct in deep_layer  for ct in others)

    if has_non_neuron:
        return "Tier0_Universal"
    if has_upper_layer:
        return "Tier1_Neuron_Identity"
    if has_deep_layer:
        return "Tier2_DeepLayer_Identity"
    return "Tier3_LayerVET_Specific"


results = {}   # {dataset: {percentile: DataFrame}}

for dataset, traj_dict_all, max_h, n_tp in [
    ("cortex",  cortex_traj, HAMMING_CORTEX, len(CORTEX_TIMEPOINTS)),
    ("spinal",  spinal_traj, HAMMING_SPINAL, len(SPINAL_TIMEPOINTS)),
]:
    results[dataset] = {}
    for pct in PERCENTILES:
        print(f"\n{'='*60}")
        print(f"Dataset: {dataset}  |  Percentile: {pct}th  |  Max Hamming: ≤{max_h}/{n_tp}")
        print('='*60)

        if dataset == "cortex":
            all_trajs = traj_dict_all[pct]   # {cell_type: {gene: traj}}
            if LAYER_V_ET not in all_trajs:
                print(f"[SKIP] {LAYER_V_ET} not in cortex cell types")
                continue
            lvet_trajs = all_trajs[LAYER_V_ET]
        else:
            # Spinal: single trajectory per gene (tissue-level)
            # Treat spinal_traj[pct] as one "cell type" track
            # Specificity analysis uses cortex cell types as reference
            lvet_trajs = traj_dict_all[pct]   # {gene: traj}
            all_trajs  = {"spinal_tissue": lvet_trajs}  # placeholder

        rows = []
        for gene, lvet_traj in lvet_trajs.items():
            if dataset == "cortex":
                matching = find_matching_cell_types(
                    gene, lvet_traj, all_trajs, max_h
                )
                tier = categorize_specificity(
                    matching, LAYER_V_ET,
                    NON_NEURON_TYPES, UPPER_LAYER_TYPES, OTHER_DEEP_LAYER_TYPES
                )
            else:
                matching = ["spinal_tissue"]
                tier = "spinal_trajectory"

            # Flag biologically interesting trajectories
            has_state2 = 2 in lvet_traj   # FORCED EXPRESSION at some point
            has_state3 = 3 in lvet_traj   # FORCED SUPPRESSION at some point
            transition = any(
                lvet_traj[i] != lvet_traj[i+1]
                for i in range(len(lvet_traj)-1)
            )

            rows.append({
                "gene"             : gene,
                "trajectory"       : "→".join(str(s) for s in lvet_traj),
                "tier"             : tier,
                "n_matching_cts"   : len(matching),
                "matching_cts"     : ",".join(sorted(matching)),
                "has_forced_expr"  : has_state2,
                "has_forced_supp"  : has_state3,
                "has_transition"   : transition,
                "percentile"       : pct,
                "dataset"          : dataset,
            })

        df = pd.DataFrame(rows)
        results[dataset][pct] = df

        tier_counts = df["tier"].value_counts()
        print(tier_counts.to_string())
        print(f"\nGenes with forced expression in trajectory: {df['has_forced_expr'].sum()}")
        print(f"Genes with forced suppression in trajectory: {df['has_forced_supp'].sum()}")

## 9. Save Results

In [ ]:
for dataset in ["cortex", "spinal"]:
    for pct in PERCENTILES:
        if pct not in results.get(dataset, {}):
            continue
        df = results[dataset][pct]

        # Save full results
        path = OUT_DIR / f"{dataset}_pct{pct}_all_trajectories.csv"
        df.sort_values(["tier", "has_forced_expr"], ascending=[True, False])\
          .to_csv(path, index=False)
        print(f"Saved: {path.name}")

        # Save each tier separately
        for tier in df["tier"].unique():
            sub = df[df["tier"] == tier].copy()
            path = OUT_DIR / f"{dataset}_pct{pct}_{tier}.csv"
            sub.to_csv(path, index=False)
            print(f"  {tier}: {len(sub)} genes")

## 10. Preview: Layer V ET-Specific Trajectories with Forced States

In [ ]:
for dataset in ["cortex", "spinal"]:
    for pct in PERCENTILES:
        if pct not in results.get(dataset, {}): continue
        df = results[dataset][pct]

        tier3 = df[
            (df["tier"] == "Tier3_LayerVET_Specific") &
            (df["has_forced_expr"] | df["has_forced_supp"])
        ].copy()

        print(f"\n{'='*60}")
        print(f"{dataset.upper()} | pct={pct}th | "
              f"Layer V ET-Specific WITH forced state: {len(tier3)} genes")
        print('='*60)
        if not tier3.empty:
            display(tier3[["gene","trajectory",
                           "has_forced_expr","has_forced_supp"]]
                    .head(30).reset_index(drop=True))
        else:
            print("None found — try lowering percentile threshold or Hamming tolerance")

## 11. Final Integration: Intersect with RNA DE + Discordance Results

In [ ]:
# Load results from previous notebooks
RNA_DE_PATH    = "/home/nakagawa/datasets/LayerV_ET_results/LayerVET_specific_genes.csv"
DISCORD_PATH   = "/home/nakagawa/datasets/discordance_results/FORCED_EXPRESSION__Tier3_LayerVET_Specific.csv"
DISCORD_PATH2  = "/home/nakagawa/datasets/discordance_results/FORCED_SUPPRESSION__Tier3_LayerVET_Specific.csv"

try:
    rna_de_genes   = set(pd.read_csv(RNA_DE_PATH)["gene"])
    discord_genes  = set(pd.read_csv(DISCORD_PATH)["gene"]) | \
                     set(pd.read_csv(DISCORD_PATH2)["gene"])
    print(f"RNA DE specific genes      : {len(rna_de_genes)}")
    print(f"Discordance specific genes : {len(discord_genes)}")
except FileNotFoundError as e:
    print(f"[NOTE] Previous results not found yet: {e}")
    rna_de_genes  = set()
    discord_genes = set()

print("\n" + "="*60)
print("GOLD LIST: Tier3 trajectory ∩ RNA DE specific ∩ Discordant")
print("="*60)

for dataset in ["cortex", "spinal"]:
    for pct in PERCENTILES:
        if pct not in results.get(dataset, {}): continue
        df = results[dataset][pct]

        tier3_genes = set(
            df[df["tier"] == "Tier3_LayerVET_Specific"]["gene"]
        )

        # Triple intersection
        gold = tier3_genes & rna_de_genes & discord_genes
        # Double intersections
        traj_rna  = tier3_genes & rna_de_genes
        traj_disc = tier3_genes & discord_genes

        print(f"\n{dataset} | pct={pct}th")
        print(f"  Tier3 trajectory only          : {len(tier3_genes)}")
        print(f"  Trajectory ∩ RNA DE            : {len(traj_rna)}")
        print(f"  Trajectory ∩ Discordance       : {len(traj_disc)}")
        print(f"  GOLD (all three)               : {len(gold)}")

        if gold:
            gold_df = df[df["gene"].isin(gold)].copy()
            gold_df.to_csv(
                OUT_DIR / f"GOLD_{dataset}_pct{pct}.csv", index=False
            )
            print(f"  Gold genes: {sorted(gold)}")

## 12. Trajectory State Summary Table
Shows what fraction of genes occupy each state at each timepoint — useful sanity check.

In [ ]:
pct = PERCENTILES[0]

print(f"Spinal cord state distribution per timepoint (pct={pct}th):")
state_mat = spinal_states[pct]
summary = pd.DataFrame(index=SPINAL_TIMEPOINTS, columns=[1,2,3,4], dtype=int)

for tp in SPINAL_TIMEPOINTS:
    if tp not in state_mat.columns: continue
    counts = state_mat[tp].value_counts()
    for s in [1,2,3,4]:
        summary.loc[tp, s] = counts.get(s, 0)

summary.columns = ["State1_Active", "State2_ForcedExpr",
                   "State3_ForcedSupp", "State4_Silent"]
display(summary)

# Flag if State2/3 counts seem too low — may need to lower percentile
total_state2 = summary["State2_ForcedExpr"].sum()
total_state3 = summary["State3_ForcedSupp"].sum()
if total_state2 + total_state3 < 100:
    print("\n[NOTE] Very few forced-state genes found.")
    print("Consider trying pct=50 threshold or checking ATAC signal quality.")